Corrected source, 28 October 2026. Outputs cleared because setup, verification labels and the private audit output path changed. The prior executed record is in notebooks/executed/02b_phase1_supplement.ipynb. This notebook retrains three models for additional logging, writing temporary checkpoints separately; it does not replace submitted weights.


# 02b - Phase 1 supplement (M1-M3)

Closes three gaps of the Phase 1 evidence without touching the submitted results in `runs/classification/`:

1. **Checkpoints.** Verifies that `runs/classification/{scratch,frozen,finetune}.pt` are the checkpoints of the submitted run (they reproduce the test accuracy in `comparison.csv`) and records their SHA-256.
2. **Full learning curves.** Re-runs the three trainings with the same configuration, seed and order as notebook 02, now also logging training accuracy and validation loss (`classify.train` computes them from quantities it already has, so training itself is unchanged). Checkpoints of this rerun go to local disk only; histories and the figure go to `runs/supplement/curves/`.
3. **Identity leakage.** Oxford-IIIT Pet has no per-animal identifier. For every test image we find the most similar training image of the same breed (ImageNet ResNet-50 features, cosine similarity) and recompute the test metrics of the submitted models on the test images that have **no** close training match. If the scores were inflated by the same animal appearing in training, they would drop on that subset.
4. **Team set audit.** Checks all 401 team photos, masks, labels and boxes and renders every photo with its boxes for visual review.

All outputs: `runs/supplement/`. Run on a Colab T4 (about 45 minutes, mostly the training rerun).

In [ ]:
# Setup — mount Drive before resolving the repository.
import os, sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    try:  # W&B key from the Colab secret WANDB_API_KEY (never typed into a cell)
        from google.colab import userdata
        os.environ.setdefault("WANDB_API_KEY", userdata.get("WANDB_API_KEY"))
    except Exception as e:
        print("W&B secret not available:", type(e).__name__)

def _resolve_spv_repo(search_root: Path, explicit: str | None = None) -> Path:
    def valid(path):
        return (path / "src" / "spv").is_dir() and (path / "configs" / "config.yaml").is_file()
    if explicit:
        candidate = Path(explicit).expanduser().resolve()
        if not valid(candidate):
            raise FileNotFoundError(f"SPV_REPO is not a project root: {candidate}")
        return candidate
    candidates = set()
    for pattern in ("smart-pet-vision", "*smart-pet-vision", "*/smart-pet-vision", "*/*/smart-pet-vision"):
        candidates.update(path.resolve() for path in search_root.glob(pattern) if valid(path))
    if valid(search_root):
        candidates.add(search_root.resolve())
    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Expected one smart-pet-vision repository under {search_root}; found {len(candidates)}. "
            "Add the shared folder shortcut to My Drive, or set SPV_REPO to its exact path."
        )
    return candidates.pop()

if IN_COLAB:
    REPO = _resolve_spv_repo(Path("/content/drive/MyDrive"), os.environ.get("SPV_REPO"))
else:
    explicit = os.environ.get("SPV_REPO")
    local_root = next((path for path in (Path.cwd(), *Path.cwd().parents)
                       if (path / "src" / "spv").is_dir()), Path.cwd())
    REPO = _resolve_spv_repo(local_root, explicit)
os.chdir(REPO)
if IN_COLAB:
    !pip -q install -r requirements.txt
sys.path.insert(0, os.path.abspath("src"))

from spv.common import load_config, seed_everything, save_json, load_json
cfg = load_config()
if IN_COLAB:  # images on fast local disk (re-downloaded each session); results stay on Drive
    cfg["paths"]["raw"] = Path("/content/data/raw")
    cfg["paths"]["processed"] = Path("/content/data/processed")
seed_everything(cfg["seed"])
RUNS = Path(cfg["paths"]["runs"])
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
from spv import data, classify, supplement
import json, numpy as np, pandas as pd
df = data.ensure_data(cfg)
IMG = data.dataset_dir(cfg) / "images"
OUT = RUNS / "classification"
SUP = RUNS / "supplement"; SUP.mkdir(parents=True, exist_ok=True)
print(len(df), "images after cleaning;", df.split.value_counts().to_dict())

## 1. Checkpoints of the submitted run

In [ ]:
loaders = classify.loaders(df, IMG, cfg)
ref = pd.read_csv(OUT / "comparison.csv").set_index("model")
ckpt, preds, check = {}, {}, []
for kind in ("scratch", "frozen", "finetune"):
    p = OUT / f"{kind}.pt"
    ckpt[kind] = {"file": f"smart-pet-vision/runs/classification/{kind}.pt",
                  "bytes": p.stat().st_size, "sha256": supplement.sha256(p)}
    model = classify.load_model(kind, cfg, p)
    logits, TEST_Y = classify.predict_logits(model, loaders["test"])
    preds[kind] = logits.argmax(1)
    acc = float((preds[kind] == TEST_Y).mean())
    sub = float(ref.loc[kind, "test_accuracy"])
    check.append({"model": kind, "test_accuracy_now": acc, "test_accuracy_submitted": sub,
                  "correct_count_difference": int(round(abs(acc - sub) * len(TEST_Y))),
                  "per_image_agreement_verified": False})
    del model; torch.cuda.empty_cache()
with open(SUP / "checkpoint_sha256.txt", "w") as f:
    for k, v in ckpt.items():
        f.write(f"{v['sha256']}  {v['file']}\n")
save_json({"checkpoints": ckpt, "reproduces_submitted_test_accuracy": check}, SUP / "checkpoints.json")
pd.DataFrame(check)

## 2. Full learning curves (rerun with training accuracy and validation loss)

In [ ]:
from pathlib import Path
CK = Path("/content/supp_ckpt") if IN_COLAB else SUP / "_ckpt_tmp"   # rerun checkpoints stay off Drive
(SUP / "curves").mkdir(parents=True, exist_ok=True)
seed_everything(cfg["seed"])                       # same seed and order as notebook 02
loaders = classify.loaders(df, IMG, cfg)
hist, rerun_info = {}, []
for kind in ("scratch", "frozen", "finetune"):
    rerun_info.append(classify.train(kind, loaders, cfg, CK, run_name=f"curves-{kind}-2026-10-28"))
    hist[kind] = pd.read_csv(CK / f"{kind}_history.csv")
    hist[kind].to_csv(SUP / "curves" / f"{kind}_history_full.csv", index=False)
submitted = {r["kind"]: r["best_val_macro_f1"] for r in load_json(OUT / "train_info.json")}
cmp = pd.DataFrame([{"model": r["kind"], "best_val_macro_f1_rerun": r["best_val_macro_f1"],
                     "best_val_macro_f1_submitted": submitted[r["kind"]]} for r in rerun_info])
cmp.to_csv(SUP / "curves" / "rerun_vs_submitted.csv", index=False)
supplement.plot_full_curves({k: hist[k] for k in ("scratch", "finetune")},
                            SUP / "curves" / "learning_curves_full.png",
                            "Validation split (n = 1,106); input 224 x 224; argmax decision, no threshold; "
                            "training metrics are running averages over augmented batches in train mode.")
supplement.plot_full_curves({"frozen": hist["frozen"]}, SUP / "curves" / "learning_curves_frozen.png",
                            "Frozen backbone; validation split; input 224 x 224; argmax, no threshold.")
for k, h in hist.items():
    last = h.iloc[-1]
    print(f"{k}: final train acc {last.train_acc:.3f}, val acc {last.val_acc:.3f}, "
          f"train loss {last.train_loss:.3f}, val loss {last.val_loss:.3f}, "
          f"min val loss at epoch {int(h.loc[h.val_loss.idxmin(), 'epoch'])}")
cmp

## 3. Identity leakage: do test scores depend on a close training match?

In [ ]:
sub = df[df.split.isin(["train", "test"])].reset_index(drop=True)
feats = supplement.embed(sub, IMG, cfg)
nn_df = supplement.nearest_train_neighbours(sub, feats)
assert (nn_df.label.values == TEST_Y).all(), "test order differs from the test loader"
nn_df.to_csv(SUP / "identity_nearest_train.csv", index=False)
eff = supplement.identity_effect(nn_df, {"scratch": preds["scratch"], "finetune": preds["finetune"]})
eff.to_csv(SUP / "identity_effect.csv", index=False)
supplement.pair_sheet(nn_df, IMG, SUP / "identity_top_pairs.jpg", n=40)
q = nn_df.nn_sim.quantile([0.5, 0.9, 0.99, 1.0]).round(3).to_dict()
save_json({"nn_sim_quantiles": q, "effect": eff.to_dict("records")}, SUP / "identity.json")
print("similarity quantiles:", q)
eff

Open `runs/supplement/identity_top_pairs.jpg`: the 40 most similar test/train pairs, for a visual check of whether they show the same animal.

## 4. Team-collected set: full audit

In [ ]:
team = supplement.audit_team(Path(cfg["paths"]["team"]), Path(cfg["paths"]["team"]) / "audit_02b")
save_json(team, SUP / "team_audit.json")
{k: (v if not isinstance(v, list) or len(v) < 15 else f"{len(v)} items") for k, v in team.items()}

In [ ]:
summary = {"checkpoints": load_json(SUP / "checkpoints.json"),
           "curves_rerun_vs_submitted": cmp.to_dict("records"),
           "identity": load_json(SUP / "identity.json"),
           "team_audit": team}
save_json(summary, SUP / "supplement_summary.json")
print("written:", sorted(str(p.relative_to(RUNS)) for p in SUP.rglob("*") if p.is_file() and "_ckpt_tmp" not in str(p)))

In [ ]:
# SHA-256 of every file of the repository as it sits on the Drive (for the submission manifest)
rows = []
for p in sorted(Path(".").rglob("*")):
    if p.is_file() and "supplement" not in p.parts and ".ipynb_checkpoints" not in p.parts and "__pycache__" not in p.parts:
        rows.append(f"{supplement.sha256(p)}  smart-pet-vision/{p.as_posix()}")
(SUP / "repo_sha256.txt").write_text("\n".join(rows) + "\n")
print(len(rows), "files hashed")